# SupplyShield AI — Advanced Feature Engineering

This notebook transforms the validated supply-chain dataset into a production-oriented feature set for anomaly detection, supplier risk modelling, NLP risk analysis, and downstream WebShield integration.

The pipeline is designed to preserve the original observations while creating explainable numerical and textual risk signals.

In [1]:
# ============================================================
# SUPPLYSHIELD AI
# MEMBER 2 — ADVANCED FEATURE ENGINEERING
# ============================================================

from __future__ import annotations

import json
import re
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42

# Display configuration
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 80)

print("=" * 80)
print("SUPPLYSHIELD AI — ADVANCED FEATURE ENGINEERING")
print("=" * 80)
print("Environment initialized successfully.")

SUPPLYSHIELD AI — ADVANCED FEATURE ENGINEERING
Environment initialized successfully.


In [2]:
# ============================================================
# DATASET DISCOVERY
# ============================================================

PROJECT_ROOT = Path.cwd()

# If notebook is executed from member2/notebooks, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

# Candidate dataset locations
DATASET_CANDIDATES = [
    PROJECT_ROOT / "member1" / "supply-webshield" / "data" / "processed" / "unified_supply_data.json",
    PROJECT_ROOT / "member1" / "supply-webshield" / "data" / "processed" / "unified_supply_data.csv",
    PROJECT_ROOT / "data" / "processed" / "unified_supply_data.json",
    PROJECT_ROOT / "data" / "processed" / "unified_supply_data.csv",
    PROJECT_ROOT / "member2" / "data" / "processed" / "unified_supply_data.json",
    PROJECT_ROOT / "member2" / "data" / "processed" / "unified_supply_data.csv",
]

DATASET_PATH = None

for candidate in DATASET_CANDIDATES:
    if candidate.exists():
        DATASET_PATH = candidate
        break

if DATASET_PATH is None:
    print("Project root:", PROJECT_ROOT)
    print("\nAvailable candidate locations:")
    for path in DATASET_CANDIDATES:
        print(" -", path)
    raise FileNotFoundError(
        "Unified supply-chain dataset could not be located. "
        "Please verify the dataset path."
    )

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset path : {DATASET_PATH}")
print(f"Dataset type : {DATASET_PATH.suffix}")

Project root : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI
Dataset path : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member1\supply-webshield\data\processed\unified_supply_data.json
Dataset type : .json


In [3]:
# ============================================================
# LOAD DATA
# ============================================================

def load_supply_dataset(path: Path) -> pd.DataFrame:
    """
    Load the unified SupplyShield dataset from JSON or CSV.

    Supports:
    - JSON list of dictionaries
    - JSON dictionary containing records
    - CSV
    """

    if path.suffix.lower() == ".json":
        with open(path, "r", encoding="utf-8") as file:
            payload = json.load(file)

        if isinstance(payload, list):
            records = payload

        elif isinstance(payload, dict):
            # Attempt to identify a record-list inside the JSON object.
            records = None

            for key in ["data", "records", "items", "results"]:
                if isinstance(payload.get(key), list):
                    records = payload[key]
                    break

            if records is None:
                records = [payload]

        else:
            raise ValueError("Unsupported JSON structure.")

        df = pd.DataFrame(records)

    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

    else:
        raise ValueError(f"Unsupported dataset format: {path.suffix}")

    return df


df = load_supply_dataset(DATASET_PATH)

print("=" * 80)
print("DATASET LOADED")
print("=" * 80)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

DATASET LOADED
Rows    : 303
Columns : 15

Columns:
['source', 'title', 'company', 'supplier', 'product', 'event', 'location', 'price', 'currency', 'availability', 'rating', 'review', 'date', 'url', 'timestamp']


,source,title,company,supplier,product,event,location,price,currency,availability,rating,review,date,url,timestamp
0,DeoDap,Wall Mount Mop Holder – No-Slide Grip for Home & Garage | GlimmerHome,GlimmerHome,None,Wall Mount Mop Holder – No-Slide Grip for Home & Garage,None,None,190.0,INR,None,4.60,131 reviews 57 reviews 57 reviews 110 reviews 110 reviews 149 reviews 149 re...,None,https://deodap.in/products/hardware-tool-multifunction-wall-mount-garage-hol...,2026-08-19T18:04:23.947119+00:00
1,DeoDap,Compact TianMu Tool Set – Essential 9-Piece Repair Kit for Home,DeoDap,None,Compact TianMu Tool Set – Essential 9-Piece Repair Kit for Home,None,None,80.0,INR,None,4.80,100 reviews 57 reviews 57 reviews 110 reviews 110 reviews 149 reviews 149 re...,None,https://deodap.in/products/compact-tianmu-combination-tool-set-9-pc-set,2026-08-19T18:04:23.947243+00:00
2,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),DeoDap,None,Manual Wall Fastening Nail Gun Tool Set (1 Set),None,None,375.0,INR,None,4.69,100 reviews 57 reviews 57 reviews 110 reviews 110 reviews 149 reviews 149 re...,None,https://deodap.in/products/manual-wall-fastening-nail-gun-tool-set-1-set,2026-08-19T18:04:23.947290+00:00
3,DeoDap,Mini Precision Screwdriver Set – Compact & Multi-Purpose Repair Tool,DeoDap,None,Mini Precision Screwdriver Set – Compact & Multi-Purpose Repair Tool,None,None,37.0,INR,None,4.48,170 reviews 405 reviews 405 reviews 147 reviews 147 reviews 636 reviews 636 ...,None,https://deodap.in/products/mini-precision-screwdriver-bit-set-with-magnetic-...,2026-08-19T18:04:23.947323+00:00
4,DeoDap,Electric Drill Machine – Compact & Powerful 280W for Versatile Tasks,DeoDap,None,Electric Drill Machine – Compact & Powerful 280W for Versatile Tasks,None,None,1.0,INR,None,4.71,102 reviews 104 reviews 104 reviews 200 reviews 200 reviews 53 reviews 53 re...,None,https://deodap.in/products/high-performance-electric-drill-machine-280w-1-pc,2026-08-19T18:04:23.947350+00:00


In [4]:
# ============================================================
# SCHEMA STANDARDIZATION
# ============================================================

# Normalize column names
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Remove accidental duplicate columns
df = df.loc[:, ~df.columns.duplicated()].copy()

# Expected schema from Member 1's unified dataset
EXPECTED_COLUMNS = [
    "source",
    "title",
    "company",
    "supplier",
    "product",
    "event",
    "location",
    "price",
    "currency",
    "availability",
    "rating",
    "review",
    "date",
    "url",
    "timestamp",
]

# Create missing columns rather than failing immediately.
for column in EXPECTED_COLUMNS:
    if column not in df.columns:
        df[column] = np.nan

# Preserve the expected order while retaining any additional columns.
remaining_columns = [
    column for column in df.columns
    if column not in EXPECTED_COLUMNS
]

df = df[EXPECTED_COLUMNS + remaining_columns]

print("Standardized schema:")
print(df.columns.tolist())

print("\nShape:", df.shape)

Standardized schema:
['source', 'title', 'company', 'supplier', 'product', 'event', 'location', 'price', 'currency', 'availability', 'rating', 'review', 'date', 'url', 'timestamp']

Shape: (303, 15)


In [5]:
# ============================================================
# DATA TYPE NORMALIZATION
# ============================================================

TEXT_COLUMNS = [
    "source",
    "title",
    "company",
    "supplier",
    "product",
    "event",
    "location",
    "currency",
    "availability",
    "review",
    "url",
]

for column in TEXT_COLUMNS:
    if column in df.columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
            .replace({
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "null": pd.NA,
                "N/A": pd.NA,
                "NA": pd.NA,
            })
        )

# Numeric conversion
for column in ["price", "rating"]:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

# Date conversion
for column in ["date", "timestamp"]:
    if column in df.columns:
        df[column] = pd.to_datetime(
            df[column],
            errors="coerce",
            utc=True
        )

# Remove impossible ratings
df.loc[
    ~df["rating"].between(0, 5, inclusive="both"),
    "rating"
] = np.nan

# Remove impossible negative prices
df.loc[
    df["price"] < 0,
    "price"
] = np.nan

# Stable observation identifier
df.insert(
    0,
    "record_id",
    [f"SS-{i:06d}" for i in range(1, len(df) + 1)]
)

print("=" * 80)
print("CLEANING COMPLETE")
print("=" * 80)

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

CLEANING COMPLETE
Shape: (303, 16)

Missing values:
event           303
location        303
supplier        303
date            303
rating          258
review          258
price            48
currency         48
availability     16
product           0
title             0
company           0
record_id         0
source            0
url               0
timestamp         0
dtype: int64


## Numerical Intelligence Features

The next stage derives explainable market, price, availability, rating, and review signals from the raw observations.

In [6]:
# ============================================================
# PRICE INTELLIGENCE FEATURES
# ============================================================

# Global price statistics
price_median = df["price"].median()
price_mean = df["price"].mean()
price_std = df["price"].std()

# Robust statistics
price_q1 = df["price"].quantile(0.25)
price_q3 = df["price"].quantile(0.75)
price_iqr = price_q3 - price_q1

# Log-transformed price
df["price_log"] = np.log1p(df["price"].clip(lower=0))

# Price deviation from global median
if pd.notna(price_median) and price_median != 0:
    df["price_deviation_pct"] = (
        (df["price"] - price_median)
        / price_median
        * 100
    )
else:
    df["price_deviation_pct"] = 0.0

# Robust price z-score using median and IQR
if pd.notna(price_iqr) and price_iqr > 0:
    df["price_robust_zscore"] = (
        (df["price"] - price_median)
        / price_iqr
    )
else:
    df["price_robust_zscore"] = 0.0

# Standard z-score
if pd.notna(price_std) and price_std > 0:
    df["price_zscore"] = (
        (df["price"] - price_mean)
        / price_std
    )
else:
    df["price_zscore"] = 0.0

# Price percentile
df["price_percentile"] = (
    df["price"]
    .rank(pct=True)
)

# IQR-based outlier flag
if pd.notna(price_iqr) and price_iqr > 0:
    lower_bound = price_q1 - 1.5 * price_iqr
    upper_bound = price_q3 + 1.5 * price_iqr

    df["price_iqr_outlier"] = (
        (df["price"] < lower_bound)
        |
        (df["price"] > upper_bound)
    ).astype(int)
else:
    df["price_iqr_outlier"] = 0

# Product-level relative price
product_price_median = (
    df.groupby("product", dropna=False)["price"]
    .transform("median")
)

df["product_price_deviation_pct"] = np.where(
    product_price_median.notna() & (product_price_median != 0),
    (
        (df["price"] - product_price_median)
        / product_price_median
        * 100
    ),
    0.0
)

# Supplier-level relative price
supplier_price_median = (
    df.groupby("supplier", dropna=False)["price"]
    .transform("median")
)

df["supplier_price_deviation_pct"] = np.where(
    supplier_price_median.notna() & (supplier_price_median != 0),
    (
        (df["price"] - supplier_price_median)
        / supplier_price_median
        * 100
    ),
    0.0
)

print("Price feature engineering completed.")

PRICE_FEATURES = [
    "price_log",
    "price_deviation_pct",
    "price_robust_zscore",
    "price_zscore",
    "price_percentile",
    "price_iqr_outlier",
    "product_price_deviation_pct",
    "supplier_price_deviation_pct",
]

display(
    df[
        ["record_id", "price"] + PRICE_FEATURES
    ].head(10)
)

Price feature engineering completed.


,record_id,price,price_log,price_deviation_pct,price_robust_zscore,price_zscore,price_percentile,price_iqr_outlier,product_price_deviation_pct,supplier_price_deviation_pct
0,SS-000001,190.0,5.252273,-80.000000,-0.043603,-0.129943,0.284314,0,0.0,-80.000000
1,SS-000002,80.0,4.394449,-91.578947,-0.049914,-0.130011,0.121569,0,0.0,-91.578947
2,SS-000003,375.0,5.929589,-60.526316,-0.032989,-0.129827,0.415686,0,0.0,-60.526316
3,SS-000004,37.0,3.637586,-96.105263,-0.052381,-0.130038,0.074510,0,0.0,-96.105263
4,SS-000005,1.0,0.693147,-99.894737,-0.054446,-0.130061,0.015686,0,0.0,-99.894737
5,SS-000006,102.0,4.634729,-89.263158,-0.048652,-0.129998,0.150980,0,0.0,-89.263158
6,SS-000007,136.0,4.919981,-85.684211,-0.046701,-0.129976,0.192157,0,0.0,-85.684211
7,SS-000008,165.0,5.111988,-82.631579,-0.045037,-0.129958,0.241176,0,0.0,-82.631579
8,SS-000009,207.0,5.337538,-78.210526,-0.042628,-0.129932,0.317647,0,0.0,-78.210526
9,SS-000010,1.0,0.693147,-99.894737,-0.054446,-0.130061,0.015686,0,0.0,-99.894737


In [7]:
# ============================================================
# AVAILABILITY INTELLIGENCE
# ============================================================

availability_text = (
    df["availability"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

def classify_availability(value: str) -> str:
    """
    Convert raw availability text into a standardized state.
    """
    if not value:
        return "unknown"

    if any(
        token in value
        for token in [
            "out of stock",
            "unavailable",
            "sold out",
            "not available",
            "currently unavailable"
        ]
    ):
        return "out_of_stock"

    if any(
        token in value
        for token in [
            "low stock",
            "limited stock",
            "few left",
            "only"
        ]
    ):
        return "low_stock"

    if any(
        token in value
        for token in [
            "in stock",
            "available",
            "ready"
        ]
    ):
        return "in_stock"

    if any(
        token in value
        for token in [
            "preorder",
            "pre-order",
            "coming soon"
        ]
    ):
        return "preorder"

    return "other"


df["availability_state"] = availability_text.apply(
    classify_availability
)

availability_score_map = {
    "in_stock": 0.00,
    "low_stock": 0.60,
    "out_of_stock": 1.00,
    "preorder": 0.50,
    "other": 0.40,
    "unknown": 0.50,
}

df["availability_risk_score"] = (
    df["availability_state"]
    .map(availability_score_map)
    .fillna(0.50)
)

# Text quality features
df["availability_text_length"] = (
    availability_text.str.len()
)

df["availability_missing_flag"] = (
    df["availability"].isna().astype(int)
)

print("Availability states:")
print(
    df["availability_state"]
    .value_counts(dropna=False)
)

display(
    df[
        [
            "record_id",
            "availability",
            "availability_state",
            "availability_risk_score",
            "availability_missing_flag"
        ]
    ].head(15)
)

Availability states:
availability_state
other      287
unknown     16
Name: count, dtype: int64


,record_id,availability,availability_state,availability_risk_score,availability_missing_flag
0,SS-000001,<NA>,unknown,0.5,1
1,SS-000002,<NA>,unknown,0.5,1
2,SS-000003,<NA>,unknown,0.5,1
3,SS-000004,<NA>,unknown,0.5,1
4,SS-000005,<NA>,unknown,0.5,1
5,SS-000006,<NA>,unknown,0.5,1
6,SS-000007,<NA>,unknown,0.5,1
7,SS-000008,<NA>,unknown,0.5,1
8,SS-000009,<NA>,unknown,0.5,1
9,SS-000010,<NA>,unknown,0.5,1


In [8]:
# ============================================================
# REVIEW + RATING INTELLIGENCE
# ============================================================

review_text = (
    df["review"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Basic review text characteristics
df["review_length"] = review_text.str.len()

df["review_word_count"] = (
    review_text
    .str.split()
    .str.len()
    .fillna(0)
)

df["review_char_count"] = (
    review_text.str.len()
)

df["review_exclamation_count"] = (
    review_text.str.count(r"!")
)

df["review_question_count"] = (
    review_text.str.count(r"\?")
)

df["review_uppercase_ratio"] = (
    review_text.apply(
        lambda text: (
            sum(char.isupper() for char in text)
            / max(sum(char.isalpha() for char in text), 1)
        )
    )
)

df["review_missing_flag"] = (
    df["review"].isna().astype(int)
)

# Rating quality
df["rating_missing_flag"] = (
    df["rating"].isna().astype(int)
)

# Normalize rating to 0–1
df["rating_normalized"] = (
    df["rating"] / 5.0
)

# Rating risk
df["rating_risk_score"] = (
    1 - df["rating_normalized"]
).clip(0, 1)

# Combined review-quality indicator
df["review_quality_signal"] = (
    np.log1p(df["review_word_count"])
    * (1 - df["review_missing_flag"])
)

print("Review/rating features generated.")

display(
    df[
        [
            "record_id",
            "rating",
            "rating_normalized",
            "rating_risk_score",
            "review_length",
            "review_word_count",
            "review_exclamation_count",
            "review_uppercase_ratio"
        ]
    ].head(10)
)

Review/rating features generated.


,record_id,rating,rating_normalized,rating_risk_score,review_length,review_word_count,review_exclamation_count,review_uppercase_ratio
0,SS-000001,4.60,0.920,0.080,249,42,0,0.000000
1,SS-000002,4.80,0.960,0.040,249,42,0,0.000000
2,SS-000003,4.69,0.938,0.062,249,42,0,0.000000
3,SS-000004,4.48,0.896,0.104,249,42,0,0.000000
4,SS-000005,4.71,0.942,0.058,241,42,0,0.025806
5,SS-000006,3.89,0.778,0.222,218,38,0,0.014599
6,SS-000007,4.71,0.942,0.058,246,42,0,0.013245
7,SS-000008,4.73,0.946,0.054,246,42,0,0.013245
8,SS-000009,4.23,0.846,0.154,249,42,0,0.000000
9,SS-000010,4.48,0.896,0.104,246,42,0,0.013245


## NLP-Oriented Risk Features

Text fields are transformed into lightweight, explainable supply-chain risk indicators. A dedicated NLP model will be added later; these features provide interpretable signals for the anomaly and risk engines.

In [9]:
# ============================================================
# SUPPLY-CHAIN NLP SIGNAL ENGINE
# ============================================================

# Combine relevant text fields
TEXT_SOURCE_COLUMNS = [
    "title",
    "event",
    "review",
    "availability"
]

for column in TEXT_SOURCE_COLUMNS:
    if column not in df.columns:
        df[column] = ""

combined_text = (
    df[TEXT_SOURCE_COLUMNS]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

df["combined_text"] = combined_text

# ------------------------------------------------------------
# Risk vocabulary
# ------------------------------------------------------------

DISRUPTION_TERMS = [
    "shutdown",
    "factory closure",
    "production halt",
    "production delay",
    "supply shortage",
    "shortage",
    "supply disruption",
    "disruption",
    "delay",
    "delayed",
    "backlog",
    "strike",
    "labor shortage",
    "transport delay",
    "shipping delay",
    "logistics issue",
    "port closure",
    "warehouse issue",
    "inventory shortage",
    "stock shortage",
    "recall",
    "supplier failure",
    "supplier disruption",
]

NEGATIVE_TERMS = [
    "problem",
    "issue",
    "failed",
    "failure",
    "poor",
    "bad",
    "damaged",
    "broken",
    "complaint",
    "fraud",
    "scam",
    "fake",
    "counterfeit",
    "unreliable",
    "warning",
    "critical",
    "severe",
    "risk",
]

URGENCY_TERMS = [
    "urgent",
    "immediately",
    "critical",
    "emergency",
    "severe",
    "high risk",
    "warning",
    "alert",
]

COUNTERFEIT_TERMS = [
    "fake",
    "counterfeit",
    "replica",
    "imitation",
    "unauthorized",
    "knockoff",
    "fraudulent",
]

def count_terms(text: str, terms: List[str]) -> int:
    """
    Count keyword/phrase occurrences in normalized text.
    """
    if not isinstance(text, str):
        return 0

    count = 0

    for term in terms:
        count += len(
            re.findall(
                rf"\b{re.escape(term)}\b",
                text
            )
        )

    return count


df["disruption_keyword_count"] = combined_text.apply(
    lambda x: count_terms(x, DISRUPTION_TERMS)
)

df["negative_keyword_count"] = combined_text.apply(
    lambda x: count_terms(x, NEGATIVE_TERMS)
)

df["urgency_keyword_count"] = combined_text.apply(
    lambda x: count_terms(x, URGENCY_TERMS)
)

df["counterfeit_keyword_count"] = combined_text.apply(
    lambda x: count_terms(x, COUNTERFEIT_TERMS)
)

# Binary indicators
df["disruption_signal_flag"] = (
    df["disruption_keyword_count"] > 0
).astype(int)

df["negative_signal_flag"] = (
    df["negative_keyword_count"] > 0
).astype(int)

df["urgency_signal_flag"] = (
    df["urgency_keyword_count"] > 0
).astype(int)

df["counterfeit_signal_flag"] = (
    df["counterfeit_keyword_count"] > 0
).astype(int)

# Text risk score
df["text_risk_score"] = (
    0.45 * np.clip(
        df["disruption_keyword_count"] / 3,
        0,
        1
    )
    +
    0.25 * np.clip(
        df["negative_keyword_count"] / 5,
        0,
        1
    )
    +
    0.20 * np.clip(
        df["urgency_keyword_count"] / 2,
        0,
        1
    )
    +
    0.10 * np.clip(
        df["counterfeit_keyword_count"] / 2,
        0,
        1
    )
)

df["text_risk_score"] = (
    df["text_risk_score"]
    .clip(0, 1)
)

print("NLP-oriented features generated.")

display(
    df[
        [
            "record_id",
            "disruption_keyword_count",
            "negative_keyword_count",
            "urgency_keyword_count",
            "counterfeit_keyword_count",
            "text_risk_score"
        ]
    ].head(15)
)

NLP-oriented features generated.


,record_id,disruption_keyword_count,negative_keyword_count,urgency_keyword_count,counterfeit_keyword_count,text_risk_score
0,SS-000001,0,0,0,0,0.0
1,SS-000002,0,0,0,0,0.0
2,SS-000003,0,0,0,0,0.0
3,SS-000004,0,0,0,0,0.0
4,SS-000005,0,0,0,0,0.0
5,SS-000006,0,0,0,0,0.0
6,SS-000007,0,0,0,0,0.0
7,SS-000008,0,0,0,0,0.0
8,SS-000009,0,0,0,0,0.0
9,SS-000010,0,0,0,0,0.0


In [10]:
# ============================================================
# SUPPLIER INTELLIGENCE
# ============================================================

# Normalize supplier names
df["supplier_normalized"] = (
    df["supplier"]
    .fillna("unknown_supplier")
    .astype(str)
    .str.lower()
    .str.strip()
)

# Supplier observation count
df["supplier_observation_count"] = (
    df.groupby("supplier_normalized")["record_id"]
    .transform("count")
)

# Supplier price statistics
df["supplier_median_price"] = (
    df.groupby("supplier_normalized")["price"]
    .transform("median")
)

df["supplier_mean_price"] = (
    df.groupby("supplier_normalized")["price"]
    .transform("mean")
)

df["supplier_price_std"] = (
    df.groupby("supplier_normalized")["price"]
    .transform("std")
)

# Supplier rating
df["supplier_mean_rating"] = (
    df.groupby("supplier_normalized")["rating"]
    .transform("mean")
)

# Supplier availability risk
df["supplier_mean_availability_risk"] = (
    df.groupby("supplier_normalized")["availability_risk_score"]
    .transform("mean")
)

# Supplier NLP risk
df["supplier_mean_text_risk"] = (
    df.groupby("supplier_normalized")["text_risk_score"]
    .transform("mean")
)

# Supplier anomaly-related signals
df["supplier_disruption_frequency"] = (
    df.groupby("supplier_normalized")["disruption_signal_flag"]
    .transform("mean")
)

df["supplier_negative_frequency"] = (
    df.groupby("supplier_normalized")["negative_signal_flag"]
    .transform("mean")
)

# Missing-data ratio at supplier level
supplier_missing_ratio = (
    df.groupby("supplier_normalized")
    .apply(
        lambda group: group.isna().mean().mean(),
        include_groups=False
    )
)

df["supplier_data_quality_risk"] = (
    df["supplier_normalized"]
    .map(supplier_missing_ratio)
    .fillna(0)
)

print("Supplier-level features generated.")

SUPPLIER_FEATURES = [
    "supplier_observation_count",
    "supplier_median_price",
    "supplier_mean_price",
    "supplier_price_std",
    "supplier_mean_rating",
    "supplier_mean_availability_risk",
    "supplier_mean_text_risk",
    "supplier_disruption_frequency",
    "supplier_negative_frequency",
    "supplier_data_quality_risk",
]

display(
    df[
        ["supplier_normalized"] + SUPPLIER_FEATURES
    ].drop_duplicates("supplier_normalized")
    .head(20)
)

Supplier-level features generated.


,supplier_normalized,supplier_observation_count,supplier_median_price,supplier_mean_price,supplier_price_std,supplier_mean_rating,supplier_mean_availability_risk,supplier_mean_text_risk,supplier_disruption_frequency,supplier_negative_frequency,supplier_data_quality_risk
0,unknown_supplier,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734


In [11]:
# ============================================================
# SOURCE-LEVEL INTELLIGENCE
# ============================================================

df["source_normalized"] = (
    df["source"]
    .fillna("unknown_source")
    .astype(str)
    .str.lower()
    .str.strip()
)

# Source observation count
df["source_observation_count"] = (
    df.groupby("source_normalized")["record_id"]
    .transform("count")
)

# Source average rating
df["source_mean_rating"] = (
    df.groupby("source_normalized")["rating"]
    .transform("mean")
)

# Source availability risk
df["source_mean_availability_risk"] = (
    df.groupby("source_normalized")["availability_risk_score"]
    .transform("mean")
)

# Source text risk
df["source_mean_text_risk"] = (
    df.groupby("source_normalized")["text_risk_score"]
    .transform("mean")
)

# Source disruption frequency
df["source_disruption_frequency"] = (
    df.groupby("source_normalized")["disruption_signal_flag"]
    .transform("mean")
)

# Source completeness
SOURCE_REQUIRED_COLUMNS = [
    "title",
    "company",
    "supplier",
    "product",
    "event",
    "location",
    "price",
    "availability",
    "rating",
    "review",
]

df["record_missingness_ratio"] = (
    df[SOURCE_REQUIRED_COLUMNS]
    .isna()
    .mean(axis=1)
)

df["record_completeness_score"] = (
    1 - df["record_missingness_ratio"]
)

print("Source-level intelligence generated.")

display(
    df[
        [
            "source_normalized",
            "source_observation_count",
            "source_mean_rating",
            "source_mean_availability_risk",
            "source_mean_text_risk",
            "source_disruption_frequency"
        ]
    ].drop_duplicates("source_normalized")
)

Source-level intelligence generated.


,source_normalized,source_observation_count,source_mean_rating,source_mean_availability_risk,source_mean_text_risk,source_disruption_frequency
0,deodap,16,4.566250,0.5,0.00000,0.0
16,tradeindia,247,NaN,0.4,0.00081,0.0
263,meesho,40,3.993103,0.4,0.00000,0.0


In [12]:
# ============================================================
# TEMPORAL FEATURES
# ============================================================

# Prefer timestamp; fall back to date
effective_datetime = df["timestamp"].copy()

effective_datetime = effective_datetime.fillna(
    df["date"]
)

df["event_year"] = (
    effective_datetime.dt.year
)

df["event_month"] = (
    effective_datetime.dt.month
)

df["event_day"] = (
    effective_datetime.dt.day
)

df["event_day_of_week"] = (
    effective_datetime.dt.dayofweek
)

df["event_week_of_year"] = (
    effective_datetime.dt.isocalendar().week.astype("float")
)

df["event_hour"] = (
    effective_datetime.dt.hour
)

# Cyclic encoding
df["month_sin"] = np.sin(
    2 * np.pi * df["event_month"].fillna(0) / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["event_month"].fillna(0) / 12
)

df["day_of_week_sin"] = np.sin(
    2 * np.pi * df["event_day_of_week"].fillna(0) / 7
)

df["day_of_week_cos"] = np.cos(
    2 * np.pi * df["event_day_of_week"].fillna(0) / 7
)

print("Temporal features generated.")

Temporal features generated.


In [13]:
# ============================================================
# COMPOSITE RISK SIGNALS
# ============================================================

# Price anomaly signal
price_anomaly_signal = (
    np.abs(
        df["price_robust_zscore"]
    ) / 3
).clip(0, 1)

# Availability signal
availability_signal = (
    df["availability_risk_score"]
    .clip(0, 1)
)

# Rating signal
rating_signal = (
    df["rating_risk_score"]
    .fillna(0.5)
    .clip(0, 1)
)

# NLP signal
nlp_signal = (
    df["text_risk_score"]
    .fillna(0)
    .clip(0, 1)
)

# Data-quality penalty
data_quality_penalty = (
    1 - df["record_completeness_score"]
).clip(0, 1)

# Supplier-level signal
supplier_signal = (
    0.35 * df["supplier_mean_availability_risk"].fillna(0.5)
    +
    0.35 * df["supplier_mean_text_risk"].fillna(0)
    +
    0.30 * df["supplier_disruption_frequency"].fillna(0)
).clip(0, 1)

# Preliminary explainable signal score
df["preliminary_risk_signal"] = (
    0.25 * price_anomaly_signal
    +
    0.20 * availability_signal
    +
    0.15 * rating_signal
    +
    0.25 * nlp_signal
    +
    0.10 * supplier_signal
    +
    0.05 * data_quality_penalty
).clip(0, 1)

# Convert to 0–100 for human interpretation
df["preliminary_risk_score"] = (
    df["preliminary_risk_signal"] * 100
).round(2)

def risk_band(score: float) -> str:
    if score >= 75:
        return "HIGH"
    elif score >= 50:
        return "MEDIUM"
    elif score >= 25:
        return "LOW"
    return "MINIMAL"


df["preliminary_risk_band"] = (
    df["preliminary_risk_score"]
    .apply(risk_band)
)

print("Composite explainable risk signals generated.")

display(
    df[
        [
            "record_id",
            "preliminary_risk_score",
            "preliminary_risk_band",
            "price_robust_zscore",
            "availability_risk_score",
            "text_risk_score",
            "supplier_mean_text_risk"
        ]
    ].sort_values(
        "preliminary_risk_score",
        ascending=False
    ).head(20)
)

Composite explainable risk signals generated.


,record_id,preliminary_risk_score,preliminary_risk_band,price_robust_zscore,availability_risk_score,text_risk_score,supplier_mean_text_risk
42,SS-000043,45.67,LOW,3.273092,0.4,0.05,0.00066
57,SS-000058,44.42,LOW,5.682731,0.4,0.00,0.00066
56,SS-000057,44.42,LOW,8.551348,0.4,0.00,0.00066
75,SS-000076,44.42,LOW,6.543316,0.4,0.00,0.00066
66,SS-000067,44.42,LOW,16.583477,0.4,0.00,0.00066
49,SS-000050,44.42,LOW,20.025818,0.4,0.00,0.00066
64,SS-000065,44.42,LOW,3.043603,0.4,0.00,0.00066
55,SS-000056,44.42,LOW,11.419966,0.4,0.00,0.00066
122,SS-000123,44.42,LOW,6.715433,0.4,0.00,0.00066
127,SS-000128,44.42,LOW,880.037292,0.4,0.00,0.00066


## ML Feature Matrix

The following stage separates raw descriptive fields from numerical features suitable for anomaly detection and future machine-learning pipelines.

In [14]:
# ============================================================
# ML FEATURE MATRIX
# ============================================================

ML_FEATURES = [
    # Price
    "price",
    "price_log",
    "price_deviation_pct",
    "price_robust_zscore",
    "price_zscore",
    "price_percentile",
    "price_iqr_outlier",
    "product_price_deviation_pct",
    "supplier_price_deviation_pct",

    # Availability
    "availability_risk_score",
    "availability_text_length",
    "availability_missing_flag",

    # Rating / review
    "rating",
    "rating_normalized",
    "rating_risk_score",
    "review_length",
    "review_word_count",
    "review_char_count",
    "review_exclamation_count",
    "review_question_count",
    "review_uppercase_ratio",
    "review_missing_flag",
    "rating_missing_flag",
    "review_quality_signal",

    # NLP
    "disruption_keyword_count",
    "negative_keyword_count",
    "urgency_keyword_count",
    "counterfeit_keyword_count",
    "disruption_signal_flag",
    "negative_signal_flag",
    "urgency_signal_flag",
    "counterfeit_signal_flag",
    "text_risk_score",

    # Supplier
    "supplier_observation_count",
    "supplier_median_price",
    "supplier_mean_price",
    "supplier_price_std",
    "supplier_mean_rating",
    "supplier_mean_availability_risk",
    "supplier_mean_text_risk",
    "supplier_disruption_frequency",
    "supplier_negative_frequency",
    "supplier_data_quality_risk",

    # Source
    "source_observation_count",
    "source_mean_rating",
    "source_mean_availability_risk",
    "source_mean_text_risk",
    "source_disruption_frequency",

    # Data quality
    "record_missingness_ratio",
    "record_completeness_score",

    # Temporal
    "event_year",
    "event_month",
    "event_day",
    "event_day_of_week",
    "event_week_of_year",
    "event_hour",
    "month_sin",
    "month_cos",
    "day_of_week_sin",
    "day_of_week_cos",

    # Composite
    "preliminary_risk_signal",
    "preliminary_risk_score",
]

# Keep only features that actually exist
ML_FEATURES = [
    feature
    for feature in ML_FEATURES
    if feature in df.columns
]

X = df[ML_FEATURES].copy()

# Replace infinite values
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

# Numeric conversion
for column in X.columns:
    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )

# Median imputation for model-ready numerical features
X = X.fillna(
    X.median(numeric_only=True)
)

# Any remaining missing values are safely filled with zero
X = X.fillna(0)

print("=" * 80)
print("ML FEATURE MATRIX CREATED")
print("=" * 80)

print("Rows:", X.shape[0])
print("Features:", X.shape[1])

display(X.head())

ML FEATURE MATRIX CREATED
Rows: 303
Features: 62


,price,price_log,price_deviation_pct,price_robust_zscore,price_zscore,price_percentile,price_iqr_outlier,product_price_deviation_pct,supplier_price_deviation_pct,availability_risk_score,availability_text_length,availability_missing_flag,rating,rating_normalized,rating_risk_score,review_length,review_word_count,review_char_count,review_exclamation_count,review_question_count,review_uppercase_ratio,review_missing_flag,rating_missing_flag,review_quality_signal,disruption_keyword_count,negative_keyword_count,urgency_keyword_count,counterfeit_keyword_count,disruption_signal_flag,negative_signal_flag,urgency_signal_flag,counterfeit_signal_flag,text_risk_score,supplier_observation_count,supplier_median_price,supplier_mean_price,supplier_price_std,supplier_mean_rating,supplier_mean_availability_risk,supplier_mean_text_risk,supplier_disruption_frequency,supplier_negative_frequency,supplier_data_quality_risk,source_observation_count,source_mean_rating,source_mean_availability_risk,source_mean_text_risk,source_disruption_frequency,record_missingness_ratio,record_completeness_score,event_year,event_month,event_day,event_day_of_week,event_week_of_year,event_hour,month_sin,month_cos,day_of_week_sin,day_of_week_cos,preliminary_risk_signal,preliminary_risk_score
0,190.0,5.252273,-80.000000,-0.043603,-0.129943,0.284314,0,0.0,-80.000000,0.5,0,1,4.60,0.920,0.080,249,42,249,0,0,0.000000,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.149842,14.98
1,80.0,4.394449,-91.578947,-0.049914,-0.130011,0.121569,0,0.0,-91.578947,0.5,0,1,4.80,0.960,0.040,249,42,249,0,0,0.000000,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.144367,14.44
2,375.0,5.929589,-60.526316,-0.032989,-0.129827,0.415686,0,0.0,-60.526316,0.5,0,1,4.69,0.938,0.062,249,42,249,0,0,0.000000,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.146257,14.63
3,37.0,3.637586,-96.105263,-0.052381,-0.130038,0.074510,0,0.0,-96.105263,0.5,0,1,4.48,0.896,0.104,249,42,249,0,0,0.000000,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.154173,15.42
4,1.0,0.693147,-99.894737,-0.054446,-0.130061,0.015686,0,0.0,-99.894737,0.5,0,1,4.71,0.942,0.058,241,42,241,0,0,0.025806,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.147445,14.74


In [15]:
# ============================================================
# FEATURE QUALITY AUDIT
# ============================================================

feature_audit = pd.DataFrame({
    "feature": X.columns,
    "dtype": X.dtypes.astype(str).values,
    "missing_count": X.isna().sum().values,
    "missing_percentage": (
        X.isna().mean().values * 100
    ),
    "unique_values": [
        X[column].nunique()
        for column in X.columns
    ],
    "mean": [
        X[column].mean()
        for column in X.columns
    ],
    "std": [
        X[column].std()
        for column in X.columns
    ],
    "min": [
        X[column].min()
        for column in X.columns
    ],
    "max": [
        X[column].max()
        for column in X.columns
    ],
})

feature_audit = (
    feature_audit
    .sort_values(
        "std",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 80)
print("FEATURE QUALITY AUDIT")
print("=" * 80)

display(feature_audit)

FEATURE QUALITY AUDIT


,feature,dtype,missing_count,missing_percentage,unique_values,mean,std,min,max
0,price,float64,0,0.0,206,1.751140e+05,1.467889e+06,5.000000e-01,1.534000e+07
1,price_deviation_pct,float64,0,0.0,206,1.833306e+04,1.545146e+05,-9.994737e+01,1.614637e+06
2,supplier_price_deviation_pct,float64,0,0.0,206,1.833306e+04,1.545146e+05,-9.994737e+01,1.614637e+06
3,price_robust_zscore,float64,0,0.0,206,9.992199e+00,8.421622e+01,-5.447504e-02,8.800373e+02
4,source_observation_count,int64,0,0.0,3,2.074752e+02,8.327700e+01,1.600000e+01,2.470000e+02
5,review_char_count,int64,0,0.0,14,1.513201e+01,5.468727e+01,0.000000e+00,2.490000e+02
6,review_length,int64,0,0.0,14,1.513201e+01,5.468727e+01,0.000000e+00,2.490000e+02
7,product_price_deviation_pct,float64,0,0.0,17,2.345026e-17,9.999122e+00,-9.512195e+01,9.512195e+01
8,review_word_count,int64,0,0.0,4,2.683168e+00,9.358140e+00,0.000000e+00,4.200000e+01
9,preliminary_risk_score,float64,0,0.0,134,2.351733e+01,9.041952e+00,1.111000e+01,4.567000e+01


In [16]:
# ============================================================
# FEATURE CORRELATION ANALYSIS
# ============================================================

correlation_matrix = X.corr(
    numeric_only=True
)

# Identify highly correlated feature pairs
HIGH_CORRELATION_THRESHOLD = 0.90

correlation_pairs = []

columns = correlation_matrix.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        correlation_value = correlation_matrix.iloc[i, j]

        if abs(correlation_value) >= HIGH_CORRELATION_THRESHOLD:
            correlation_pairs.append({
                "feature_1": columns[i],
                "feature_2": columns[j],
                "correlation": correlation_value
            })

high_correlation_df = pd.DataFrame(
    correlation_pairs
)

print("=" * 80)
print("HIGH-CORRELATION FEATURE PAIRS")
print("=" * 80)

if high_correlation_df.empty:
    print("No feature pairs exceeded the 0.90 correlation threshold.")
else:
    display(
        high_correlation_df.sort_values(
            "correlation",
            key=lambda s: s.abs(),
            ascending=False
        )
    )

HIGH-CORRELATION FEATURE PAIRS


,feature_1,feature_2,correlation
27,availability_missing_flag,source_mean_rating,1.000000
22,availability_text_length,source_mean_rating,-1.000000
9,price_robust_zscore,supplier_price_deviation_pct,1.000000
5,price_deviation_pct,price_robust_zscore,1.000000
1,price,price_robust_zscore,1.000000
50,record_missingness_ratio,record_completeness_score,-1.000000
17,availability_risk_score,source_mean_availability_risk,1.000000
29,rating,rating_normalized,1.000000
23,availability_text_length,source_mean_availability_risk,-1.000000
42,review_missing_flag,rating_missing_flag,1.000000


In [17]:
# ============================================================
# FINAL FEATURE DATASET
# ============================================================

IDENTIFIER_COLUMNS = [
    "record_id",
    "source",
    "title",
    "company",
    "supplier",
    "product",
    "event",
    "location",
    "currency",
    "url",
]

CONTEXT_COLUMNS = [
    "availability_state",
    "supplier_normalized",
    "source_normalized",
    "preliminary_risk_band",
]

# Ensure columns exist
IDENTIFIER_COLUMNS = [
    column
    for column in IDENTIFIER_COLUMNS
    if column in df.columns
]

CONTEXT_COLUMNS = [
    column
    for column in CONTEXT_COLUMNS
    if column in df.columns
]

final_feature_dataset = pd.concat(
    [
        df[
            IDENTIFIER_COLUMNS + CONTEXT_COLUMNS
        ].reset_index(drop=True),

        X.reset_index(drop=True)
    ],
    axis=1
)

# Remove duplicate columns
final_feature_dataset = (
    final_feature_dataset
    .loc[
        :,
        ~final_feature_dataset.columns.duplicated()
    ]
)

print("=" * 80)
print("FINAL FEATURE DATASET")
print("=" * 80)

print(
    f"Rows    : {final_feature_dataset.shape[0]:,}"
)

print(
    f"Columns : {final_feature_dataset.shape[1]:,}"
)

display(
    final_feature_dataset.head()
)

FINAL FEATURE DATASET
Rows    : 303
Columns : 76


,record_id,source,title,company,supplier,product,event,location,currency,url,availability_state,supplier_normalized,source_normalized,preliminary_risk_band,price,price_log,price_deviation_pct,price_robust_zscore,price_zscore,price_percentile,price_iqr_outlier,product_price_deviation_pct,supplier_price_deviation_pct,availability_risk_score,availability_text_length,availability_missing_flag,rating,rating_normalized,rating_risk_score,review_length,review_word_count,review_char_count,review_exclamation_count,review_question_count,review_uppercase_ratio,review_missing_flag,rating_missing_flag,review_quality_signal,disruption_keyword_count,negative_keyword_count,urgency_keyword_count,counterfeit_keyword_count,disruption_signal_flag,negative_signal_flag,urgency_signal_flag,counterfeit_signal_flag,text_risk_score,supplier_observation_count,supplier_median_price,supplier_mean_price,supplier_price_std,supplier_mean_rating,supplier_mean_availability_risk,supplier_mean_text_risk,supplier_disruption_frequency,supplier_negative_frequency,supplier_data_quality_risk,source_observation_count,source_mean_rating,source_mean_availability_risk,source_mean_text_risk,source_disruption_frequency,record_missingness_ratio,record_completeness_score,event_year,event_month,event_day,event_day_of_week,event_week_of_year,event_hour,month_sin,month_cos,day_of_week_sin,day_of_week_cos,preliminary_risk_signal,preliminary_risk_score
0,SS-000001,DeoDap,Wall Mount Mop Holder – No-Slide Grip for Home & Garage | GlimmerHome,GlimmerHome,<NA>,Wall Mount Mop Holder – No-Slide Grip for Home & Garage,<NA>,<NA>,INR,https://deodap.in/products/hardware-tool-multifunction-wall-mount-garage-hol...,unknown,unknown_supplier,deodap,MINIMAL,190.0,5.252273,-80.000000,-0.043603,-0.129943,0.284314,0,0.0,-80.000000,0.5,0,1,4.60,0.920,0.080,249,42,249,0,0,0.000000,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.149842,14.98
1,SS-000002,DeoDap,Compact TianMu Tool Set – Essential 9-Piece Repair Kit for Home,DeoDap,<NA>,Compact TianMu Tool Set – Essential 9-Piece Repair Kit for Home,<NA>,<NA>,INR,https://deodap.in/products/compact-tianmu-combination-tool-set-9-pc-set,unknown,unknown_supplier,deodap,MINIMAL,80.0,4.394449,-91.578947,-0.049914,-0.130011,0.121569,0,0.0,-91.578947,0.5,0,1,4.80,0.960,0.040,249,42,249,0,0,0.000000,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.144367,14.44
2,SS-000003,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),DeoDap,<NA>,Manual Wall Fastening Nail Gun Tool Set (1 Set),<NA>,<NA>,INR,https://deodap.in/products/manual-wall-fastening-nail-gun-tool-set-1-set,unknown,unknown_supplier,deodap,MINIMAL,375.0,5.929589,-60.526316,-0.032989,-0.129827,0.415686,0,0.0,-60.526316,0.5,0,1,4.69,0.938,0.062,249,42,249,0,0,0.000000,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.146257,14.63
3,SS-000004,DeoDap,Mini Precision Screwdriver Set – Compact & Multi-Purpose Repair Tool,DeoDap,<NA>,Mini Precision Screwdriver Set – Compact & Multi-Purpose Repair Tool,<NA>,<NA>,INR,https://deodap.in/products/mini-precision-screwdriver-bit-set-with-magnetic-...,unknown,unknown_supplier,deodap,MINIMAL,37.0,3.637586,-96.105263,-0.052381,-0.130038,0.074510,0,0.0,-96.105263,0.5,0,1,4.48,0.896,0.104,249,42,249,0,0,0.000000,0,0,3.7612,0,0,0,0,0,0,0,0,0.0,303,950.0,207897.846196,1.598460e+06,4.196889,0.405281,0.00066,0.0,0.013201,0.150734,16,4.56625,0.5,0.0,0.0,0.4,0.6,2026,8,19,2,34.0,18,-0.866025,-0.5,0.974928,-0.222521,0.154173,15.42
4,SS-000005,DeoDap,Electric Drill Machine – Compact & Powerful 280W for Versatile Tasks,DeoDap

In [18]:
# ============================================================
# FEATURE ENGINEERING SUMMARY
# ============================================================

numeric_feature_columns = (
    final_feature_dataset
    .select_dtypes(include=np.number)
    .columns
)

summary = pd.DataFrame({
    "feature_count": [len(ML_FEATURES)],
    "dataset_rows": [len(final_feature_dataset)],
    "dataset_columns": [final_feature_dataset.shape[1]],
    "numeric_features": [len(numeric_feature_columns)],
    "missing_cells": [
        int(final_feature_dataset.isna().sum().sum())
    ],
    "duplicate_rows": [
        int(final_feature_dataset.duplicated().sum())
    ],
})

display(summary)

print("\nRisk-band distribution:")
print(
    final_feature_dataset[
        "preliminary_risk_band"
    ].value_counts(dropna=False)
)

,feature_count,dataset_rows,dataset_columns,numeric_features,missing_cells,duplicate_rows
0,62,303,76,62,957,0



Risk-band distribution:
preliminary_risk_band
MINIMAL    236
LOW         67
Name: count, dtype: int64


In [19]:
# ============================================================
# SAVE FEATURE ENGINEERING ARTIFACTS
# ============================================================

OUTPUT_DIR = (
    PROJECT_ROOT
    / "member2"
    / "outputs"
    / "feature_engineering"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Main feature dataset
FEATURE_DATASET_PATH = (
    OUTPUT_DIR
    / "supplyshield_feature_dataset.csv"
)

final_feature_dataset.to_csv(
    FEATURE_DATASET_PATH,
    index=False,
    encoding="utf-8"
)

# Feature list
FEATURE_LIST_PATH = (
    OUTPUT_DIR
    / "ml_feature_list.json"
)

with open(
    FEATURE_LIST_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        ML_FEATURES,
        file,
        indent=2
    )

# Feature audit
FEATURE_AUDIT_PATH = (
    OUTPUT_DIR
    / "feature_quality_audit.csv"
)

feature_audit.to_csv(
    FEATURE_AUDIT_PATH,
    index=False,
    encoding="utf-8"
)

# Metadata
metadata = {
    "project": "SupplyShield AI",
    "pipeline": "advanced_feature_engineering",
    "rows": int(final_feature_dataset.shape[0]),
    "columns": int(final_feature_dataset.shape[1]),
    "ml_features": int(len(ML_FEATURES)),
    "source_dataset": str(DATASET_PATH),
    "random_state": RANDOM_STATE,
    "high_correlation_threshold": HIGH_CORRELATION_THRESHOLD,
}

METADATA_PATH = (
    OUTPUT_DIR
    / "feature_engineering_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metadata,
        file,
        indent=2
    )

print("=" * 80)
print("FEATURE ENGINEERING ARTIFACTS SAVED")
print("=" * 80)

print("Feature dataset :", FEATURE_DATASET_PATH)
print("Feature list    :", FEATURE_LIST_PATH)
print("Quality audit   :", FEATURE_AUDIT_PATH)
print("Metadata        :", METADATA_PATH)

FEATURE ENGINEERING ARTIFACTS SAVED
Feature dataset : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\feature_engineering\supplyshield_feature_dataset.csv
Feature list    : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\feature_engineering\ml_feature_list.json
Quality audit   : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\feature_engineering\feature_quality_audit.csv
Metadata        : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\feature_engineering\feature_engineering_metadata.json


In [20]:
# ============================================================
# FINAL PIPELINE VALIDATION
# ============================================================

assert len(final_feature_dataset) > 0, (
    "Final feature dataset is empty."
)

assert len(ML_FEATURES) > 0, (
    "No ML features were generated."
)

assert final_feature_dataset["record_id"].is_unique, (
    "record_id values are not unique."
)

assert not X.isna().any().any(), (
    "ML feature matrix still contains missing values."
)

assert not np.isinf(X.to_numpy()).any(), (
    "ML feature matrix contains infinite values."
)

assert FEATURE_DATASET_PATH.exists(), (
    "Feature dataset was not saved."
)

print("=" * 80)
print("FEATURE ENGINEERING VALIDATION PASSED")
print("=" * 80)

print(f"Records                 : {len(final_feature_dataset):,}")
print(f"Total output columns    : {final_feature_dataset.shape[1]:,}")
print(f"ML features             : {len(ML_FEATURES):,}")
print(f"Missing ML values       : {X.isna().sum().sum():,}")
print(f"Duplicate record IDs    : {final_feature_dataset['record_id'].duplicated().sum():,}")
print(f"Output file             : {FEATURE_DATASET_PATH}")
print()
print("READY FOR ANOMALY-DETECTION ML.")

FEATURE ENGINEERING VALIDATION PASSED
Records                 : 303
Total output columns    : 76
ML features             : 62
Missing ML values       : 0
Duplicate record IDs    : 0
Output file             : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\feature_engineering\supplyshield_feature_dataset.csv

READY FOR ANOMALY-DETECTION ML.


## Feature Engineering Complete

The processed dataset now contains explainable numerical, temporal, supplier, source, availability, review, NLP-oriented, and composite risk features.

The generated feature matrix will be consumed by the anomaly-detection and risk-modelling pipeline.